# Train Fraud-Detection Models 1-4 on Colab GPU

Runs **Tasks 4.2 / 4.3 / 4.4 / 4.5** (Transformer-only, Transformer+GraphSAGE, +GAT, +ST-HGNN) on a free T4 GPU using the project's existing `src/train.py` unchanged.

All four models train here on the GPU. Local CPU was ~10 min/epoch for the transformer alone (~8.5 h for a full 50-epoch run) and far worse for the GNNs (which re-run a 590k-node graph every batch), so the whole comparison study moved to the free GPU.

**Before you start:**
1. Upload `colab_package.zip` to your Google Drive (in **My Drive**, the root — or note the path and edit the `ZIP` variable below).
2. `Runtime > Change runtime type > Hardware accelerator = T4 GPU` (free tier), then Save.
3. Run the cells **top to bottom**.

Outputs (the 4 checkpoints + the completed `comparison.csv`) are copied back to `My Drive/colab_outputs/` at the end so you can pull them into your local project.

## 1. Confirm the GPU is enabled

In [ ]:
!nvidia-smi
# If this errors or shows no GPU: Runtime > Change runtime type > T4 GPU, then rerun.

## 2. Install PyTorch Geometric
Colab already ships a CUDA build of PyTorch, so we only add PyG. We pin **2.8.0** (the version `graph.pt` was saved with) so the `HeteroData` object unpickles cleanly. The convs we use (SAGEConv/GATConv/HeteroConv) work on pure PyG — no `torch-scatter`/`torch-sparse` compile needed.

In [ ]:
!pip install -q torch_geometric==2.8.0
import torch, torch_geometric
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
print('torch_geometric', torch_geometric.__version__)
assert torch.cuda.is_available(), 'No GPU! Enable T4 GPU via Runtime > Change runtime type.'

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Unzip the package into the Colab runtime
Edit `ZIP` if you didn't put `colab_package.zip` in the root of My Drive.

In [ ]:
import os, zipfile

ZIP = '/content/drive/MyDrive/colab_package.zip'   # <-- adjust path if needed
DEST = '/content/ntcc'
assert os.path.exists(ZIP), f'zip not found at {ZIP} — upload colab_package.zip to My Drive.'

os.makedirs(DEST, exist_ok=True)
with zipfile.ZipFile(ZIP) as zf:
    zf.extractall(DEST)
os.chdir(DEST)
print('unzipped to', DEST)
!ls -R /content/ntcc | head -40

## 5. Sanity check: device should be CUDA
`config.get_device()` auto-detects the GPU, so `train.py` will train on CUDA with mixed precision automatically.

In [ ]:
!cd /content/ntcc && python -c "import config; print('device =', config.get_device())"

## 6. Task 4.2 — Model 1: Transformer only
Saves `models/m1_transformer.pt` and appends a TEST-metrics row to `results/comparison.csv`. No graph is used for this model (sequence view only).

In [ ]:
!cd /content/ntcc && python src/train.py --model m1

## 7. Task 4.3 — Model 2: Transformer + GraphSAGE
Saves `models/m2_sage.pt`.

In [ ]:
!cd /content/ntcc && python src/train.py --model m2

## 8. Task 4.4 — Model 3: Transformer + GAT
Saves `models/m3_gat.pt`.

In [ ]:
!cd /content/ntcc && python src/train.py --model m3

## 9. Task 4.5 — Model 4: Transformer + ST-HGNN
Saves `models/m4_sthgnn.pt`. If you hit a CUDA fp16/autocast error in the GAT attention, rerun this cell with `--no-amp` appended.

In [ ]:
!cd /content/ntcc && python src/train.py --model m4
# Fallback if an AMP error occurs:
# !cd /content/ntcc && python src/train.py --model m4 --no-amp

## 10. Review the comparison table

In [ ]:
import pandas as pd
pd.read_csv('/content/ntcc/results/comparison.csv')

## 11. Save outputs back to Google Drive
Copies the 4 checkpoints + the completed `comparison.csv` to `My Drive/colab_outputs/`.

In [ ]:
import os, shutil

OUT = '/content/drive/MyDrive/colab_outputs'
os.makedirs(OUT + '/models', exist_ok=True)
for m in ['m1_transformer.pt', 'm2_sage.pt', 'm3_gat.pt', 'm4_sthgnn.pt']:
    src = f'/content/ntcc/models/{m}'
    if os.path.exists(src):
        shutil.copy(src, f'{OUT}/models/{m}')
        print('saved', m)
    else:
        print('MISSING (did its cell run?):', m)
shutil.copy('/content/ntcc/results/comparison.csv', f'{OUT}/comparison.csv')
print('\nAll outputs in Drive ->', OUT)

## 12. Pull results back to your local project
On your machine, from `My Drive/colab_outputs/`:
- copy `models/m1_transformer.pt`, `m2_sage.pt`, `m3_gat.pt`, `m4_sthgnn.pt` into your local `models/`
- copy `comparison.csv` into your local `results/comparison.csv` (it contains all **four** GPU-trained model rows = the full comparison table).

That completes Tasks 4.2–4.5. Next is Phase 5 (evaluate.py + plots), which runs fine locally on CPU.